In [7]:
# All Packages
import os
import pandas as pd
import numpy as np
from llama_cloud_services import LlamaExtract
from dotenv import load_dotenv
from typing import Optional, Type, List
from pydantic import ValidationError, BaseModel, Field
from scipy.stats import mode
import itertools

from BS_Schema_251017 import make_StatementOfFinancialPosition_model

### Extraction Setup

In [8]:
# Import Class
SFP = make_StatementOfFinancialPosition_model(2024)
print(SFP.__name__)
print(list(SFP.__fields__.keys()))

StatementOfFinancialPosition_2024
['cash_and_short_term_investments_unrestricted', 'cash_and_short_term_investments_restricted', 'accounts_receivable', 'pledges_receivable', 'government_grants_and_other_receivables', 'loans_receivable', 'all_receivables', 'accumulated_depreciation_bs', 'accumulated_amortization_bs', 'accumulated_depreciation_notes', 'accumulated_amortization_notes', 'rou_assets_finance_lease_bs', 'rou_assets_finance_lease_notes', 'net_fixed_assets_raw', 'rou_assets_operating_lease_bs', 'rou_assets_operating_lease_notes', 'long_term_investments', 'cash_surrender_value_life_insurance', 'total_assets', 'short_term_debt', 'accounts_payable', 'adjustments_to_accounts_payable', 'deferred_revenue', 'def_rev_mixed', 'asset_retirement_obligations', 'finance_lease_liability_bs', 'finance_lease_liability_notes', 'long_term_debt_labeled', 'other_long_term_debt_obligations', 'backup_total_long_term_debt', 'operating_lease_liability_bs', 'operating_lease_liability_notes', 'pension_l

/var/folders/yc/4xwzhl2x2yq_cvnvnqf6hv580000gn/T/ipykernel_19937/2593354945.py:4: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  print(list(SFP.__fields__.keys()))


In [ ]:
PDF_ROOT = "university_pdfs_hy_3"                   # folder containing PDFs to process
OUTPUT_ROOT = "output_Balance_Sheet_DS"             # folder to save results
os.makedirs(OUTPUT_ROOT, exist_ok=True)             # create output folder if it doesn't exist
AGENT_ID = "bcb15a18-67ac-4772-9965-2654ecaff88c"   # existing agent ID
load_dotenv()
Year = 2024
extractor = LlamaExtract(api_key='llx-jlyOr0ZQwOzn0BPYgzggmo6mFShTLTsxdrXGviOaBf6IDnHG',project_id = '8c10e62e-3810-4193-915d-d2d11105826d')# LlamaExtract(project_id = '8c10e62e-3810-4193-915d-d2d11105826d')

# #uncomment the below line if you are creating the agent for the first time
# agent = extractor.create_agent(name = "balance-sheet-parser-v1", data_schema=SFP)

agent = extractor.get_agent(id = AGENT_ID)

# uncomment the following lines if you updated the schema - can always run these two lines to ensure the latest schema is used
agent.data_schema = SFP
agent.save()
# agent = extractor.get_agent(id = AGENT_ID)

Extracting files: 100%|██████████| 1/1 [01:41<00:00, 101.49s/it]


### Functions

In [10]:
def process_school(school_name, school_dir):        # process all PDFs in a school's directory
    combined   = {}
    first_keys = None

    for fname in sorted(os.listdir(school_dir)):
        if not fname.lower().endswith(".pdf"):
            continue
        path = os.path.join(school_dir, fname)
        print(f"Extracting data from {fname}")
        try:
            run  = agent.extract(path)
            data = run.data or {}
            if first_keys is None:
                first_keys = list(data.keys())
                combined  = {k: None for k in first_keys}
            for k, v in data.items():
                if v not in (None, "", []):
                    combined[k] = v
        except Exception as err:
            print(f"Skipped {fname}: {err}")

    if first_keys:
        df = pd.DataFrame.from_dict(combined, orient="index", columns=["2024-25"])
        df.index.name = "Metric"
        outfile = os.path.join(OUTPUT_ROOT, f"{school_name}.xlsx")
        df.to_excel(outfile)
        print(f"Saved output to {outfile}")
    else:
        print(f"No PDF data found for {school_name}")

In [ ]:
def run_extraction_once():              # this output file will be formatted as one row per extracted file
    results = {}

    for school in sorted(os.listdir(PDF_ROOT)):
        school_dir = os.path.join(PDF_ROOT, school)
        if not os.path.isdir(school_dir):
            continue

        pdf_files = [
            f for f in sorted(os.listdir(school_dir))
            if f.lower().endswith(".pdf")
        ]

        if len(pdf_files) == 1:
            to_read = pdf_files
        else:
            financial_only = [f for f in pdf_files if "financial" in f.lower()]
            to_read = financial_only if financial_only else pdf_files

        for fname in to_read:
            path = os.path.join(school_dir, fname)
            print(f"Extracting data from {school}/{fname}")
            try:
                run  = agent.extract(path)
                data = run.data or {}
                results[(school, fname)] = data
            except Exception as err:
                print(f"Skipped {fname}: {err}")

        break

    return results

In [ ]:
def safe_mode(x):                    # in this all-in-one version notebook, we find mode first, then post process certain fields
    counts = x.value_counts(dropna=True)
    if counts.empty:
        return 0

    # Case 1: No unique mode (tie in counts)
    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1:].max():
        unique_vals = np.sort(x.dropna().unique())
        
        # --- Rule A: All 3 values within 1% of their average ---
        if len(unique_vals) == 3:
            avg_all = np.mean(unique_vals)
            diffs = np.abs(unique_vals - avg_all) / avg_all
            if (diffs < 0.01).all():
                return avg_all

            # --- Rule B: Any 2 of 3 values within 5% of each other ---
            for a, b in itertools.combinations(unique_vals, 2):  # all (n choose 2) pairs
                if abs(a - b) / np.mean([a, b]) < 0.05:
                    return np.mean([a, b])
                
        # --- Rule C: if no mode is found, return the latest one
        return x.iloc[-1]

    return counts.idxmax()

### Multiple Extractions

In [21]:
n = 3               # number of extraction runs to perform
all_results = []

for i in range(n):
    print(f"\n=== RUN {i+1} ===")
    results_i = run_extraction_once()
    all_results.append(results_i)


=== RUN 1 ===
Extracting data from ALFRED_UNIVERSITY/2024_Audited_Financial_Statements_for_the_year_ended_06_30_2024__364_KB_.pdf

=== RUN 2 ===
Extracting data from ALFRED_UNIVERSITY/2024_Audited_Financial_Statements_for_the_year_ended_06_30_2024__364_KB_.pdf

=== RUN 3 ===
Extracting data from ALFRED_UNIVERSITY/2024_Audited_Financial_Statements_for_the_year_ended_06_30_2024__364_KB_.pdf


In [22]:
dfs = []  # will store df1, df2, ..., dfN

for res in all_results:
    df = pd.DataFrame.from_dict(res, orient="index")                            # Build DataFrame: one row per (school, filename)
    df.index = pd.MultiIndex.from_tuples(df.index, names=["school", "file"])    # optional: give the index clearer names

    dfs.append(df)

### Aggregate Extractions

In [27]:
combined = pd.concat(dfs, keys=range(len(dfs)))

combined = combined.copy()
combined["RowType"] = "data"

combined = combined.reset_index(level=0, drop=True)

if isinstance(combined.columns, pd.MultiIndex):
    combined.columns = combined.columns.droplevel(0)

if 'school' in combined.index.names:
    combined = combined.reset_index(level='school')

combined.head()

,school,cash_and_short_term_investments_unrestricted,cash_and_short_term_investments_restricted,accounts_receivable,pledges_receivable,government_grants_and_other_receivables,loans_receivable,all_receivables,accumulated_depreciation_bs,accumulated_amortization_bs,...,swap_obligation_fmv,total_liabilities,net_assets_without_donor_restrictions,perpetual_net_assets_with_donor_restrictions,net_assets_with_donor_restrictions,noncontrolling_interest,total_net_assets,total_liabilities_and_net_assets,units_multiplier,RowType
file,,,,,,,,,,,,,,,,,,,,,
2024_Audited_Financial_Statements_for_the_year_ended_06_30_2024__364_KB_.pdf,ALFRED_UNIVERSITY,316104,18916648,38429245,6883042,9135355,7199713,72186244,None,None,...,None,97404551,155677020,101559880,198807484,None,354484504,451889055,1,data
2024_Audited_Financial_Statements_for_the_year_ended_06_30_2024__364_KB_.pdf,ALFRED_UNIVERSITY,316104,0,38429245,6883042,None,0,68276363,None,None,...,None,97404551,155677020,None,198807484,None,354484504,451889055,None,data
2024_Audited_Financial_Statements_for_the_year_ended_06_30_2024__364_KB_.pdf,ALFRED_UNIVERSITY,316104,18916648,38429245,6883042,0,22964076,68276363,None,None,...,None,97404551,155677020,101559880,198807484,None,354484504,451889055,1,data


### Calculate Mode (for all raw extracted columns)

In [ ]:
# At this point, extracted columns include:
    #    'cash_and_short_term_investments_unrestricted', 'cash_and_short_term_investments_restricted', 
    #    'accounts_receivable', 'pledges_receivable', 'government_grants_and_other_receivables', 'loans_receivable', 'all_receivables', 
    #    'accumulated_depreciation_bs', 'accumulated_amortization_bs', 'accumulated_depreciation_notes', 'accumulated_amortization_notes', 
    #    'rou_assets_finance_lease_bs', 'rou_assets_finance_lease_notes', 
    #    'net_fixed_assets_raw',
    #    'rou_assets_operating_lease_bs', 'rou_assets_operating_lease_notes',
    #    'long_term_investments', 'cash_surrender_value_life_insurance',
    #    'total_assets',
    #    'short_term_debt', 'accounts_payable', 'adjustments_to_accounts_payable', 
    #    'deferred_revenue', 'def_rev_mixed', 'asset_retirement_obligations', 
    #    'finance_lease_liability_bs', 'finance_lease_liability_notes', 
    #    'long_term_debt_labeled', 'other_long_term_debt_obligations', 'backup_total_long_term_debt',
    #    'operating_lease_liability_bs', 'operating_lease_liability_notes',
    #    'pension_liability', 'opeb_liability', 'pension_and_opeb_liability',
    #    'swap_obligation_fmv', 'total_liabilities',
    #    'current_portion_finance_lease', 'current_portion_long_term_debt', 'current_portion_operating_lease',
    #    'net_assets_without_donor_restrictions',
    #    'perpetual_net_assets_with_donor_restrictions',
    #    'net_assets_with_donor_restrictions', 'noncontrolling_interest',
    #    'total_net_assets', 'total_liabilities_and_net_assets',
    #    'year', 'units_multiplier'

In [29]:
mode_df = combined.groupby(['school'], dropna=False).agg(safe_mode).reset_index()
mode_df["RowType"] = "mode"
mode_df.head()

,school,cash_and_short_term_investments_unrestricted,cash_and_short_term_investments_restricted,accounts_receivable,pledges_receivable,government_grants_and_other_receivables,loans_receivable,all_receivables,accumulated_depreciation_bs,accumulated_amortization_bs,...,swap_obligation_fmv,total_liabilities,net_assets_without_donor_restrictions,perpetual_net_assets_with_donor_restrictions,net_assets_with_donor_restrictions,noncontrolling_interest,total_net_assets,total_liabilities_and_net_assets,units_multiplier,RowType
0,ALFRED_UNIVERSITY,316104,18916648,38429245,6883042,0,22964076,68276363,0,0,...,0,97404551,155677020,101559880,198807484,0,354484504,451889055,1,mode


### Combine Original and Mode

In [31]:
final_df = pd.concat([combined, mode_df], axis=0)

# Sort within each school, preserve stability of ordering
final_df["_sort_key"] = final_df["RowType"].map(lambda x: 1 if x == "data" else 2)
final_df = (
    final_df.sort_values(by=["school", "_sort_key"], kind="mergesort")
            .drop(columns="_sort_key")
            .reset_index(drop=True)
)

final_df.head()

,school,cash_and_short_term_investments_unrestricted,cash_and_short_term_investments_restricted,accounts_receivable,pledges_receivable,government_grants_and_other_receivables,loans_receivable,all_receivables,accumulated_depreciation_bs,accumulated_amortization_bs,...,swap_obligation_fmv,total_liabilities,net_assets_without_donor_restrictions,perpetual_net_assets_with_donor_restrictions,net_assets_with_donor_restrictions,noncontrolling_interest,total_net_assets,total_liabilities_and_net_assets,units_multiplier,RowType
0,ALFRED_UNIVERSITY,316104,18916648,38429245,6883042,9135355,7199713,72186244,None,None,...,None,97404551,155677020,101559880,198807484,None,354484504,451889055,1,data
1,ALFRED_UNIVERSITY,316104,0,38429245,6883042,None,0,68276363,None,None,...,None,97404551,155677020,None,198807484,None,354484504,451889055,None,data
2,ALFRED_UNIVERSITY,316104,18916648,38429245,6883042,0,22964076,68276363,None,None,...,None,97404551,155677020,101559880,198807484,None,354484504,451889055,1,data
3,ALFRED_UNIVERSITY,316104,18916648,38429245,6883042,0,22964076,68276363,0,0,...,0,97404551,155677020,101559880,198807484,0,354484504,451889055,1,mode


### Post-processing

In [38]:
def col(df, name):                                              # helper function to get column or zero Series if missing       
    if name in df.columns:
        return df[name].fillna(0)
    else:
        return pd.Series(0, index=df.index)

def add_plug_accounts(df: pd.DataFrame) -> pd.DataFrame:

    net_receivables_components = [                              # post-process net receivables
        "accounts_receivable",
        "pledges_receivable",
        "government_grants_and_other_receivables",
        "loans_receivable"
    ]
    df["receivables_leftover_calculated"] = df["all_receivables"] - df[net_receivables_components].fillna(0).sum(axis=1)
    df["net_receivables"] = df[net_receivables_components].fillna(0).sum(axis=1) + df["receivables_leftover_calculated"]
    
                                                                # post-process pension and opeb liability
    df["pension_and_opeb_liability"] = df["pension_and_opeb_liability"].fillna(
                col(df, "pension_liability").fillna(0) + col(df,"opeb_liability").fillna(0)
            )
    # df["pension_and_opeb_liability"] = df.get("pension_liability", 0).fillna(0) + df.get("opeb_liability", 0).fillna(0)

    df["accumulated_depreciation"] = (                          # post-process accumulated depreciation
    np.where(                                                       # Depreciation: use one of them if bs == notes to avoid double counting
        df["accumulated_depreciation_bs"].fillna(0)                     # if not, use sum of both
        == df["accumulated_depreciation_notes"].fillna(0),
        df["accumulated_depreciation_bs"].fillna(0),
        df["accumulated_depreciation_bs"].fillna(0)
        + df["accumulated_depreciation_notes"].fillna(0),
    )
    +
    np.where(                                                       # Amortization: same as above
        df["accumulated_amortization_bs"].fillna(0)
        == df["accumulated_amortization_notes"].fillna(0),
        df["accumulated_amortization_bs"].fillna(0),
        df["accumulated_amortization_bs"].fillna(0)
        + df["accumulated_amortization_notes"].fillna(0),
    )
)

    df["long_term_investments_unrestricted_and_restricted"] = ( # post-process long term investments
        col(df,"long_term_investments").fillna(0)
        + col(df,"cash_surrender_value_life_insurance").fillna(0)
    )

                                                                # post-process deferred revenue
    df["all_deferred_revenue"] = df['deferred_revenue'] - col(df,"def_rev_mixed").fillna(0) * col(df,"asset_retirement_obligations").fillna(0)

                                                                # post-process accounts payable
    df["accounts_payable_adjusted"] = col(df,"accounts_payable").fillna(0) - col(df,"adjustments_to_accounts_payable").fillna(0)
                                                               
    df["long_term_debt"] = (                                    # post-process long-term debt      
        col(df,"long_term_debt_labeled").fillna(0)
        + col(df,"other_long_term_debt_obligations").fillna(0)
        + col(df,"finance_lease_liability").fillna(0)
    )
    df["long_term_debt"] = df["long_term_debt"].where(
            df["long_term_debt"] != 0,
            col(df,"backup_total_long_term_debt").fillna(0)
    )

    pairs_to_sum = {                                            # post-process ROU assets and lease liabilities           
        "rou_assets_finance_lease": ["rou_assets_finance_lease_bs", "rou_assets_finance_lease_notes"],
        "rou_assets_operating_lease": ["rou_assets_operating_lease_bs", "rou_assets_operating_lease_notes"],
        "finance_lease_liability": ["finance_lease_liability_bs", "finance_lease_liability_notes"],
        "operating_lease_liability": ["operating_lease_liability_bs", "operating_lease_liability_notes"],
    }
    for new_col, cols_to_sum in pairs_to_sum.items():
        if len(cols_to_sum) == 1:
            df[new_col] = df[cols_to_sum[0]]                    
        else:
            col1, col2 = cols_to_sum
            df[new_col] = df.apply(
                lambda row: (
                    row[col1] if pd.isna(row[col2]) else
                    row[col2] if pd.isna(row[col1]) else
                    row[col1] if row[col1] == row[col2] else
                    row[col2]
                ),
                axis=1
            )
                                                                # post-process net fixed assets
    df["net_fixed_assets"] = df["net_fixed_assets_raw"].fillna(0) + df["rou_assets_finance_lease"].fillna(0)
    
    asset_components = [                                         # compute plug accounts: other assets, other liabilities               
        "cash_and_short_term_investments_unrestricted",
        "cash_and_short_term_investments_restricted",
        "net_receivables",
        "net_fixed_assets",
        "long_term_investments_unrestricted_and_restricted",
        "rou_assets_operating_lease"
    ]
    liability_components = [
        "short_term_debt",
        "current_portion_finance_lease",
        "current_portion_long_term_debt",
        "current_portion_operating_lease",
        "accounts_payable_adjusted",
        "all_deferred_revenue",
        "long_term_debt",
        "finance_lease_liability",
        "operating_lease_liability",
        "swap_obligation_fmv",
        "pension_and_opeb_liability"
    ]
    assets_cols = [c for c in asset_components if c in df.columns]
    liabs_cols  = [c for c in liability_components if c in df.columns]
    
    df["sum_asset_components"]     = df[assets_cols].sum(axis=1, skipna=True)
    df["sum_liability_components"] = df[liabs_cols].sum(axis=1, skipna=True)
    df["other_assets_plug"]     = df["total_assets"] - df["sum_asset_components"]
    df["other_liabilities_plug"] = df["total_liabilities"] - df["sum_liability_components"]

                                                                    # post-process expendable net assets with donor restrictions
    df["expendable_net_assets_with_donor_restrictions"] = df["net_assets_with_donor_restrictions"] - df["perpetual_net_assets_with_donor_restrictions"]
    
    return df


In [39]:
mode_df_processed = add_plug_accounts(mode_df.copy())
mode_df_processed["RowType"] = "mode"
mode_df_processed.head()

,school,cash_and_short_term_investments_unrestricted,cash_and_short_term_investments_restricted,accounts_receivable,pledges_receivable,government_grants_and_other_receivables,loans_receivable,all_receivables,accumulated_depreciation_bs,accumulated_amortization_bs,...,rou_assets_finance_lease,rou_assets_operating_lease,finance_lease_liability,operating_lease_liability,net_fixed_assets,sum_asset_components,sum_liability_components,other_assets_plug,other_liabilities_plug,expendable_net_assets_with_donor_restrictions
0,ALFRED_UNIVERSITY,316104,18916648,38429245,6883042,0,22964076,68276363,0,0,...,0,0,0,0,165137918,451280354,72580511,608701,24824040,97247604


In [ ]:
desired_order = [
    "year", 
    "cash_and_short_term_investments_unrestricted",
    "cash_and_short_term_investments_restricted",
    "net_receivables",
    "accumulated_depreciation",
    "rou_assets_finance_lease",
    "net_fixed_assets",
    "rou_assets_operating_lease", 
    "long_term_investments_unrestricted_and_restricted",
    "other_assets_plug",                                              # now we have this calculated at the end
    "total_assets", 
    "short_term_debt", "accounts_payable_adjusted", 
    "all_deferred_revenue", 
    "finance_lease_liability",
    "long_term_debt",
    "operating_lease_liability", 
    "swap_obligation_fmv", 
    "pension_and_opeb_liability", 
    "other_liabilities_plug",                                         # now we have this calculated at the end
    "total_liabilities",
    "net_assets_without_donor_restrictions", 
    "expendable_net_assets_with_donor_restrictions",
    "perpetual_net_assets_with_donor_restrictions", 
    "net_assets_with_donor_restrictions",
    "noncontrolling_interest",
    "total_net_assets", 
    "total_liabilities_and_net_assets"
]

remaining_columns = [col for col in mode_df_processed.columns if col not in desired_order]
full_column_order = desired_order + remaining_columns

mode_df_reordered = mode_df_processed[full_column_order]

In [ ]:
# save the df with mode rows plus post-processed fields
OUTPUT_FILE = "sampled_universities_bs_1025.xlsx"
mode_df_reordered.to_excel(OUTPUT_FILE)

# save the df with both extracted data and mode rows
OUTPUT_FILE2 = "extracted_and_mode_sampled_universities.xlsx"
final_df.to_excel(OUTPUT_FILE2)             